Build LLM from Scratch

In [2]:
from torch.utils.data import DataLoader

from src.GPTDatasetV1 import GPTDatasetV1

# Download the text data. Comment out the following cell if you already have the file saved locally.
file_path = "data/the-verdict.txt"
'''
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
urllib.request.urlretrieve(url, file_path)
'''

'\nimport urllib.request\nurl = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")\nurllib.request.urlretrieve(url, file_path)\n'

In [4]:
# Read the text file
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters: ", len(raw_text))
print(raw_text[:100])

Total number of characters:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [5]:
# Example tokenization
import re
text = "Hello, world! This is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world!', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [6]:
# Segments punctuation example
result = re.split(r'([,.!?]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '!', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [14]:
# remove empty strings / whitespace.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '.']


Note: You may not want to remove whitespaces if formatting is important. For example, if you want to preserve line breaks or if you are processing python code which uses whitespace for indentation.

Additionally, we leave capitalization intact to ensure proper nouns are identified. With enough data and training, the probabilities will ensure the correct capitalization is used.

In [15]:
# Let's modify to handle a more complex example
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [24]:
from src.DataProcessing import tokenizer_v1
preprocessed_tokens = tokenizer_v1(raw_text)
print(f"Token count: {len(preprocessed_tokens)}")
print(preprocessed_tokens[:30])

Token count: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [25]:
all_words = sorted(set(preprocessed_tokens))
vocab_size: int = len(all_words)
print(f"Vocab size: {vocab_size}")

Vocab size: 1130


In [26]:
# create a dictionary mapping words to integers
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(f"{i}: {item}")
    if i >= 30:
        break

0: ('!', 0)
1: ('"', 1)
2: ("'", 2)
3: ('(', 3)
4: (')', 4)
5: (',', 5)
6: ('--', 6)
7: ('.', 7)
8: (':', 8)
9: (';', 9)
10: ('?', 10)
11: ('A', 11)
12: ('Ah', 12)
13: ('Among', 13)
14: ('And', 14)
15: ('Are', 15)
16: ('Arrt', 16)
17: ('As', 17)
18: ('At', 18)
19: ('Be', 19)
20: ('Begin', 20)
21: ('Burlington', 21)
22: ('But', 22)
23: ('By', 23)
24: ('Carlo', 24)
25: ('Chicago', 25)
26: ('Claude', 26)
27: ('Come', 27)
28: ('Croft', 28)
29: ('Destroyed', 29)
30: ('Devonshire', 30)


When we want to turn token ID back into text, we need an inverse version of the vocabulary for efficient lookups in the other direction.

To ensure reproducibility, we will create a class for the tokenizer.

In [17]:
from src.Tokenizers.SimpleTokenizerV1 import SimpleTokenizerV1

tokenizer = SimpleTokenizerV1(vocab)
text = """"
It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride.
"""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [18]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [19]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

Notice: that we get a key error if there is a word found not in the vocabulary. There are three common ways to deal with this: (1) Add special context tokens such as <|unk|> or (2) Add a mask token to ignore words that are not in the vocabulary, or (3) use a better tokenizer that can break down / still represent unseen words (byte pair encoding).

In [30]:
from src.Tokenizers.SimpleTokenizerV2 import SimpleTokenizerV2
all_tokens = sorted(set(preprocessed_tokens))
all_tokens.extend(["<|endoftext|>", "<|unk|>"]) # Add special tokens
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print(f"Token count: {len(vocab)}")

Token count: 1132


Notice there are two additional token in our vocab from earlier.

In [41]:
# Now let's try out our new tokenizer.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
tokenizer = SimpleTokenizerV2(vocab)
print(text)
print(tokenizer.encode(text))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


Notice that the code no longer errors out when unknown tokens are seen.

In [42]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


Some researchers also consider additional special tokens such as:
[BOS] (beginning of sequence)
[EOS] (end of sequence)
[PAD] (padding)

Byte Pair Encoding (Alternative Tokenization Methodology)

In [44]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 5372, 13]


In [45]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace.


Note: Rather than relying on the unknown token. Byte pair encoding breaks down words that are not in it's vocabulary.

Data sampling with sliding window.
Aka codification of the self-supervised learning data set generation process.

In [47]:
enc_text = tokenizer.encode(raw_text)
print(f"Token length: {len(enc_text)}")

Token length: 5145


In [50]:
enc_sample = enc_text[50:]
contex_size = 4
x = enc_sample[:contex_size]
y = enc_sample[1:contex_size+1]
print(f"Context size: {contex_size}")
print(f"Input: {x}")
print(f"Output:     {y}")

Context size: 4
Input: [290, 4920, 2241, 287]
Output:     [4920, 2241, 287, 257]


In [51]:
for i in range(1, contex_size+1):
    contex = enc_sample[:i]
    desired = enc_sample[i]
    print(contex, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


Everything to the left of the arrow represents the input to the LLM, and the token Id on the right side represents the token the LLM is supposed to predict.

In [53]:
for i in range(1, contex_size+1):
    contex = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(contex), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


For efficient data loading, we use PyTorch's built-in Dataset and Data Loader classes.

In [54]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True,
                         drop_last=True, number_of_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=number_of_workers
    )
    return dataloader

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)